In [12]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
TARGET_COL = 'Depression'
ID_COL = 'id'

train_path = '../data/train.csv'
test_path = '../data/test.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

In [13]:
# 1. Tách feature và target
drop_cols = [ID_COL, 'Name']

X = train.drop(columns=[TARGET_COL] + drop_cols)
y = train[TARGET_COL]
X_test = test.drop(columns=drop_cols)
test_ids = test[ID_COL]

In [14]:
#2. Chuẩn hoá categorical, xử lý missing theo vai trò và thêm missing indicator
ROLE_COL = 'Working Professional or Student'
STUDENT_ONLY_NUMERIC_COLS = ['Academic Pressure', 'CGPA', 'Study Satisfaction']
WORK_ONLY_NUMERIC_COLS = ['Work Pressure', 'Job Satisfaction']
WORK_ONLY_CATEGORICAL_COLS = ['Profession']
BINARY_YES_NO_COLS = [
    'Have you ever had suicidal thoughts ?',
    'Family History of Mental Illness',
]


def clean_string_columns(df):
    df = df.copy()
    obj_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in obj_cols:
        # sklearn SimpleImputer can fail on pandas pd.NA, so keep missing as np.nan.
        df[col] = df[col].astype('object')
        df[col] = df[col].where(df[col].notna(), np.nan)
        df[col] = df[col].str.strip()
        df[col] = df[col].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    return df


def add_missing_indicators(df, cols):
    df = df.copy()
    for col in cols:
        df[f'{col}_is_missing'] = df[col].isna().astype(int)
    return df


def fill_structural_missing_by_role(df):
    df = df.copy()
    is_student = df[ROLE_COL].eq('Student')
    is_professional = df[ROLE_COL].eq('Working Professional')

    for col in STUDENT_ONLY_NUMERIC_COLS:
        if col in df.columns:
            df.loc[is_professional & df[col].isna(), col] = 0

    for col in WORK_ONLY_NUMERIC_COLS:
        if col in df.columns:
            df.loc[is_student & df[col].isna(), col] = 0

    for col in WORK_ONLY_CATEGORICAL_COLS:
        if col in df.columns:
            df.loc[is_student & df[col].isna(), col] = 'Not Applicable'

    return df


def map_yes_no_columns(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].map({'No': 0, 'Yes': 1})
    return df


def group_rare_categories(train_df, test_df, min_count=30):
    train_df = train_df.copy()
    test_df = test_df.copy()
    cat_cols = train_df.select_dtypes(include=['object', 'string']).columns.tolist()

    for col in cat_cols:
        counts = train_df[col].value_counts(dropna=True)
        keep_values = set(counts[counts >= min_count].index)

        train_mask = train_df[col].notna() & ~train_df[col].isin(keep_values)
        test_mask = test_df[col].notna() & ~test_df[col].isin(keep_values)
        train_df.loc[train_mask, col] = 'Other'
        test_df.loc[test_mask, col] = 'Other'

    return train_df, test_df


X_clean = clean_string_columns(X)
X_test_clean = clean_string_columns(X_test)

missing_cols = X_clean.columns[X_clean.isna().mean() > 0].tolist()
print('Columns with missing values before role-aware fill:', missing_cols)

X_clean = add_missing_indicators(X_clean, missing_cols)
X_test_clean = add_missing_indicators(X_test_clean, missing_cols)

X_clean = fill_structural_missing_by_role(X_clean)
X_test_clean = fill_structural_missing_by_role(X_test_clean)

X_clean = map_yes_no_columns(X_clean, BINARY_YES_NO_COLS)
X_test_clean = map_yes_no_columns(X_test_clean, BINARY_YES_NO_COLS)

X_clean, X_test_clean = group_rare_categories(X_clean, X_test_clean, min_count=30)

Columns with missing values before role-aware fill: ['Profession', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Dietary Habits', 'Degree', 'Financial Stress']


In [15]:
# 3.Chia train/val có stratify 
X_train, X_val, y_train, y_val = train_test_split(
    X_clean,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

In [16]:
# 4.Tạo Preprocessing Pipeline cho tree-based models
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('onehot', make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols),
    ],
    remainder='drop',
)

In [17]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test_clean)

In [18]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index,
)
X_val_processed_df = pd.DataFrame(
    X_val_processed,
    columns=feature_names,
    index=X_val.index,
)

In [20]:
# Chạy thử nhiều model tree-based và so sánh trong một lần chạy
from sklearn.base import clone
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.tree import DecisionTreeClassifier


optional_models = {}
missing_optional_packages = []

try:
    from xgboost import XGBClassifier

    optional_models['XGBoost'] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='logloss',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
except ImportError:
    missing_optional_packages.append('xgboost')

try:
    from lightgbm import LGBMClassifier

    optional_models['LightGBM'] = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )
except ImportError:
    missing_optional_packages.append('lightgbm')

try:
    from catboost import CatBoostClassifier

    optional_models['CatBoost'] = CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        random_seed=RANDOM_STATE,
        verbose=False,
    )
except ImportError:
    missing_optional_packages.append('catboost')


tree_models = {
    # 'Decision Tree': DecisionTreeClassifier(
    #     max_depth=8,
    #     min_samples_leaf=10,
    #     class_weight='balanced',
    #     random_state=RANDOM_STATE,
    # ),
    # 'Random Forest': RandomForestClassifier(
    #     n_estimators=300,
    #     min_samples_leaf=5,
    #     class_weight='balanced',
    #     random_state=RANDOM_STATE,
    #     n_jobs=-1,
    # ),
    # 'Extra Trees': ExtraTreesClassifier(
    #     n_estimators=300,
    #     min_samples_leaf=5,
    #     class_weight='balanced',
    #     random_state=RANDOM_STATE,
    #     n_jobs=-1,
    # ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE,
    ),
    # 'AdaBoost': AdaBoostClassifier(
    #     n_estimators=200,
    #     learning_rate=0.05,
    #     random_state=RANDOM_STATE,
    # ),
}
tree_models.update(optional_models)

if missing_optional_packages:
    print(
        'Skipped optional models because packages are not installed: '
        + ', '.join(missing_optional_packages)
    )
    print('Install them with: pip install -r ml/requirements.txt')

results = []
reports = {}
fitted_models = {}

for name, classifier in tree_models.items():
    model = Pipeline(
        steps=[
            ('preprocess', clone(preprocessor)),
            ('classifier', classifier),
        ]
    )

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    val_proba = model.predict_proba(X_val)[:, 1]

    reports[name] = classification_report(y_val, val_pred)
    fitted_models[name] = model
    results.append(
        {
            'model': name,
            'accuracy': accuracy_score(y_val, val_pred),
            'precision': precision_score(y_val, val_pred),
            'recall': recall_score(y_val, val_pred),
            'f1': f1_score(y_val, val_pred),
            'roc_auc': roc_auc_score(y_val, val_proba),
        }
    )

results_df = pd.DataFrame(results).sort_values(
    by=['roc_auc', 'f1'],
    ascending=False,
).reset_index(drop=True)

display(results_df)

best_model_name = results_df.loc[0, 'model']
best_model = fitted_models[best_model_name]

print(f'Best model by ROC-AUC: {best_model_name}')
print(reports[best_model_name])

/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,accuracy,precision,recall,f1,roc_auc
0,CatBoost,0.939161,0.846546,0.812439,0.829142,0.974264
1,XGBoost,0.938770,0.844093,0.813221,0.828369,0.974258
2,LightGBM,0.918657,0.712907,0.924702,0.805109,0.973951
3,Gradient Boosting,0.937598,0.844166,0.805202,0.824224,0.972908


Best model by ROC-AUC: CatBoost
              precision    recall  f1-score   support

           0       0.96      0.97      0.96     23027
           1       0.85      0.81      0.83      5113

    accuracy                           0.94     28140
   macro avg       0.90      0.89      0.90     28140
weighted avg       0.94      0.94      0.94     28140



In [22]:
# Tạo submission mẫu
final_model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        (
            'classifier',
            # RandomForestClassifier(
            #     n_estimators=300,
            #     min_samples_leaf=5,
            #     class_weight='balanced',
            #     random_state=RANDOM_STATE,
            #     n_jobs=-1,
            # ),
            CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            random_seed=RANDOM_STATE,
            verbose=False,
        )
        ),
    ]
)

final_model.fit(X_clean, y)
test_pred = final_model.predict(X_test_clean)

submission = pd.DataFrame({ID_COL: test_ids, TARGET_COL: test_pred})
display(submission.head())

submission.to_csv('submission.csv', index=False)

,id,Depression
0,140700,0
1,140701,0
2,140702,0
3,140703,1
4,140704,0
